In [ ]:
# # Ubuntu/Debian
# sudo apt-get install -y poppler-utils tesseract-ocr libmagic1

# # macOS
# brew install poppler tesseract libmagic

In [ ]:
import os

In [ ]:
from unstructured.partition.auto import partition

elements = partition(filename="K2_Think.pdf")

In [ ]:
# partition() does 3 things internally:
# 1. Detects file type using libmagic (or file extension as fallback)
# 2. Routes to the correct partition_xxx() function
# 3. Returns a List[Element]

In [ ]:
for item in elements:
    print(f" --- {type(item).__name__} : {item.text[:80]}")

In [ ]:
print(type(elements))         # <class 'list'>
print(len(elements))          # e.g. 42
print(type(elements[0]))      # <class 'unstructured.documents.elements.Title'>

In [ ]:
type(elements[1]).__name__

In [ ]:
type(elements[6]).__name__

In [ ]:
el = elements[0]

# ── Core fields ──────────────────────────────────────
print(el.text)                  # The actual extracted text
print(el.category)              # "Title", "NarrativeText", etc.
print(type(el).__name__)        # same as category, class name
print(el.id)                    # unique hash ID for this element
print(el.to_dict())             # full dict representation

# ── Output of el.to_dict() looks like: ───────────────
# {
#   'type':       'NarrativeText',
#   'element_id': 'a3f1c82b...',
#   'text':       'Machine learning is a branch of AI...',
#   'metadata': {
#       'filename':     'report.pdf',
#       'filetype':     'application/pdf',
#       'page_number':  1,
#       'languages':    ['eng'],
#       'coordinates':  {...},
#       'parent_id':    'b605350b...',
#   }
# }

In [ ]:
meta = el.metadata

# ── Always present (when available) ──────────────────
print(meta.filename)            # "report.pdf"
print(meta.file_directory)      # "/home/user/docs"
print(meta.filetype)            # "application/pdf"
print(meta.page_number)         # 1, 2, 3 ...
print(meta.languages)           # ['eng']
print(meta.last_modified)       # "2024-01-15T10:30:00"

# ── Hierarchy / Structure ─────────────────────────────
print(meta.parent_id)           # ID of the parent element (e.g. Title above this)
print(meta.category_depth)      # depth in document hierarchy (0 = top level)

# ── Coordinates (bounding box on page) ───────────────
print(meta.coordinates)         # bounding box object
print(meta.coordinates.to_dict()) #type: ignore
# {
#   'points': ((10,10), (10,100), (200,100), (200,10)),
#   'system': 'PixelSpace',
#   'layout_width': 850,
#   'layout_height': 1100
# }

# ── Table specific ────────────────────────────────────
print(meta.text_as_html)        # "<table><tr><td>...</td></tr></table>"
                                # only present for Table elements

# ── Email specific ────────────────────────────────────
print(meta.sent_from)           # ["sender@example.com"]
print(meta.sent_to)             # ["recipient@example.com"]
print(meta.subject)             # "Meeting Notes"
print(meta.email_message_id)    # "<unique-id@mail.com>"

# ── Image specific ────────────────────────────────────
print(meta.image_path)          # path to extracted image file
print(meta.image_mime_type)     # "image/png"

# ── Full dict ─────────────────────────────────────────
print(meta.to_dict())           # everything as a flat dictionary

In [ ]:
from unstructured.partition.auto import partition

In [ ]:
from unstructured.documents.elements import (
    Title,
    NarrativeText,
    ListItem,
    Table,
    Image,
    Header,
    Footer,
    FigureCaption,
    Address,
    EmailAddress,
    CodeSnippet,
    PageBreak,
    Text,
)

elements = partition(filename="K2Think-1-4.pdf", languages=["eng"])

# ── Filter by class type ──────────────────────────────
titles      = [e for e in elements if isinstance(e, Title)]
paragraphs  = [e for e in elements if isinstance(e, NarrativeText)]
tables      = [e for e in elements if isinstance(e, Table)]
images      = [e for e in elements if isinstance(e, Image)]
list_items  = [e for e in elements if isinstance(e, ListItem)]
code        = [e for e in elements if isinstance(e, CodeSnippet)]

# ── Filter using .category string ────────────────────
titles = [e for e in elements if e.category == "Title"]

# ── Count distribution of types ──────────────────────
from collections import Counter
dist = Counter(type(e).__name__ for e in elements)
print(dist)
# Counter({'NarrativeText': 30, 'Title': 8, 'ListItem': 6, 'Table': 2})

In [ ]:
from unstructured.partition.auto import partition
from unstructured.documents.coordinates import (
    PixelSpace,
    RelativeCoordinateSystem,
)

el = elements[2]

if el.metadata.coordinates:
    coords = el.metadata.coordinates

    print(coords.points)
    print(coords.system.width) #type: ignore
    print(coords.system.height) #type: ignore
    print(coords.system.orientation) #type: ignore

    # Convert to relative (0.0–1.0) coordinate system
    el.convert_coordinates_to_new_system(
        RelativeCoordinateSystem(),
        in_place=True
    )
    print(el.metadata.coordinates.points)

In [ ]:
# from unstructured.partition.text import partition_text

# text = """
# SPEAKER 1: Welcome to the meeting.
# SPEAKER 2: Thanks for having me.
# ACTION ITEM: Follow up by Friday.
# """

# elements = partition_text(
#     text=text,
#     regex_metadata={
#         "speaker":     r"SPEAKER \d+:",
#         "action_item": r"ACTION ITEM:",
#     }
# )

# for el in elements:
#     print(el.text)
#     # use getattr with default instead of direct access
#     regex_meta = getattr(el.metadata, 'regex_metadata', None)
#     print("regex_metadata →", regex_meta)

In [ ]:
from unstructured.partition.auto import partition
from unstructured.chunking.title import chunk_by_title

elements = partition(filename="K2Think-1-4.pdf", languages=["eng"])

chunks = chunk_by_title(
    elements,
    max_characters=1000,
    new_after_n_chars=800,
    overlap=50,
    combine_text_under_n_chars=200,
    include_orig_elements=True,   # ← attach source elements to each chunk
)

chunk = chunks[0]

# ── What's inside a chunk ─────────────────────────────
print(type(chunk).__name__)          # CompositeElement
print(chunk.text)                    # merged text of all elements in chunk
print(len(chunk.text))               # character count

# ── Source elements that made up this chunk ───────────
for source_el in chunk.metadata.orig_elements: #type: ignore
    print(f"  [{type(source_el).__name__}] {source_el.text[:60]}")

# ── Chunk metadata (inherited from first element) ─────
print(chunk.metadata.filename)
print(chunk.metadata.page_number)
print(chunk.metadata.to_dict())

In [ ]:
meta = el.metadata

# ── Always present (when available) ──────────────────
print(meta.filename)            # "report.pdf"
print(meta.file_directory)      # "/home/user/docs"
print(meta.filetype)            # "application/pdf"
print(meta.page_number)         # 1, 2, 3 ...
print(meta.languages)           # ['eng']
print(meta.last_modified)       # "2024-01-15T10:30:00"

# ── Hierarchy / Structure ─────────────────────────────
print(meta.parent_id)           # ID of the parent element (e.g. Title above this)
print(meta.category_depth)      # depth in document hierarchy (0 = top level)

# ── Coordinates (bounding box on page) ───────────────
print(meta.coordinates)         # bounding box object
print(meta.coordinates.to_dict()) #type:ignore
# {
#   'points': ((10,10), (10,100), (200,100), (200,10)),
#   'system': 'PixelSpace',
#   'layout_width': 850,
#   'layout_height': 1100
# }

# ── Table specific ────────────────────────────────────
print(meta.text_as_html)        # "<table><tr><td>...</td></tr></table>"
                                # only present for Table elements

# ── Email specific ────────────────────────────────────
print(meta.sent_from)           # ["sender@example.com"]
print(meta.sent_to)             # ["recipient@example.com"]
print(meta.subject)             # "Meeting Notes"
print(meta.email_message_id)    # "<unique-id@mail.com>"

# ── Image specific ────────────────────────────────────
print(meta.image_path)          # path to extracted image file
print(meta.image_mime_type)     # "image/png"

# ── Full dict ─────────────────────────────────────────
# print(meta.to_dict())           # everything as a flat dictionary

In [ ]:
from unstructured.chunking.title import chunk_by_title

chunks = chunk_by_title(
    elements,
    max_characters=1000,
    new_after_n_chars=800,
    overlap=50,
    combine_text_under_n_chars=200,
    include_orig_elements=True,   # ← attach source elements to each chunk
)

chunk = chunks[0]

# ── What's inside a chunk ─────────────────────────────
print(type(chunk).__name__)          # CompositeElement
print(chunk.text)                    # merged text of all elements in chunk
print(len(chunk.text))               # character count

# ── Source elements that made up this chunk ───────────
for source_el in chunk.metadata.orig_elements: #type: ignore
    print(f"  [{type(source_el).__name__}] {source_el.text[:60]}")

# ── Chunk metadata (inherited from first element) ─────
print(chunk.metadata.filename)
print(chunk.metadata.page_number)
print(chunk.metadata.to_dict())

### **Chunking**

In [ ]:
from unstructured.partition.auto import partition
from unstructured.chunking.basic import chunk_elements

In [ ]:
elements = partition(filename="attention.pdf", languages=["eng"])

In [ ]:
chunks = chunk_elements(
    elements,
    max_characters=1000,      # hard max — no chunk exceeds this
    new_after_n_chars=800,    # soft max — prefer to close chunk here
    overlap=50,               # char overlap between text-split chunks
    overlap_all=False,        # don't overlap clean semantic boundaries
    include_orig_elements=True,
)

In [ ]:
for i, chunk in enumerate(chunks):
    print(f"\n{'-'*60}")
    print(f"Chunk #{i+1} | type={type(chunk).__name__} | chars={len(chunk.text)}")
    print(chunk.text[:200])


In [ ]:
chunk = chunks[16]   # pick any chunk

print(type(chunk).__name__)       # CompositeElement / Table / TableChunk
print(chunk.id)                   # unique hash ID
print(chunk.category)             # "CompositeElement"
print(chunk.text)                 # full merged text of the chunk
print(len(chunk.text))            # character count
# print(chunk.to_dict())            # everything as a flat dict

In [ ]:
meta = chunk.metadata

print(meta.filename)              # source file name
print(meta.filetype)              # "application/pdf"
print(meta.page_number)           # page where the chunk starts
print(meta.languages)             # ['eng']
print(meta.last_modified)         # file modification date

print(meta.to_dict())             # all metadata fields as dict — safest inspection method

In [ ]:
for i, chunk in enumerate(chunks):
    orig = chunk.metadata.orig_elements or []
    type_dist = {}
    for el in orig:
        t = type(el).__name__
        type_dist[t] = type_dist.get(t, 0) + 1

    print(f"Chunk #{i+1:03d} | {type(chunk).__name__:20s} | "
          f"chars={len(chunk.text):5d} | "
          f"page={chunk.metadata.page_number} | "
          f"src_elements={len(orig)} | "
          f"types={type_dist}")


In [1]:
! uv pip install tesseract

Using Python 3.13.3 environment at: D:\PROJECTS\.venv
Audited 1 package in 39ms


In [1]:
import os
from unstructured_pytesseract import pytesseract

pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
os.environ["TESSDATA_PREFIX"] = r"C:\Program Files\Tesseract-OCR\tessdata"  # optional

In [2]:
from unstructured.partition.pdf import partition_pdf
import os

os.makedirs("./extracted_images", exist_ok=True)

elements = partition_pdf(
    filename="attention.pdf",
    strategy="hi_res",                              # required — runs layout model
    extract_image_block_types=["Image", "Table"],   # tell it what to extract
    extract_image_block_output_dir="./extracted_images",  # save to disk
    extract_image_block_to_payload=False,           # set True to get base64 inline
    infer_table_structure=True,
    hi_res_model_name="yolox",
)

Loading weights: 100%|██████████| 367/367 [00:00<00:00, 597.08it/s] 


In [3]:
from unstructured.documents.elements import Image

image_elements = [el for el in elements if el.category == "Image"]

print(f"Total elements     : {len(elements)}")
print(f"Image elements     : {len(image_elements)}")

for i, el in enumerate(image_elements):
    print(f"\nImage #{i+1}")
    print(f"  page            : {el.metadata.page_number}")
    print(f"  image_path      : {el.metadata.image_path}")       # path on disk
    print(f"  image_mime_type : {el.metadata.image_mime_type}")
    # print(f"  coordinates     : {el.metadata.coordinates}")
    print(f"  text (caption)  : {el.text}")                      # OCR text inside image if any

Total elements     : 232
Image elements     : 7

Image #1
  page            : 3
  image_path      : ./extracted_images\figure-3-1.jpg
  image_mime_type : None
  text (caption)  : Output Probabilities Add & Norm Feed Forward Add & Norm Multi-Head Attention a, Add & Norm Nx Add & Norm Feed Forward Nx | -Casda Nom] Add & Norm VWEeea Multi-Head Multi-Head Attention Attention Sy ae, Se a, Positional @ Encoding @ Positional @ q Encoding Input Embedding Inputs Outputs (shifted right) Output Embedding

Image #2
  page            : 4
  image_path      : ./extracted_images\figure-4-2.jpg
  image_mime_type : None
  text (caption)  : 

Image #3
  page            : 4
  image_path      : ./extracted_images\figure-4-3.jpg
  image_mime_type : None
  text (caption)  : EEE Scaled Dot-Product | Attention 4

Image #4
  page            : 13
  image_path      : ./extracted_images\figure-13-4.jpg
  image_mime_type : None
  text (caption)  : n £ c < c 2 2 > & oO n a= Ze i) o > s o 8 =| HPAANAAAAA Ez, 8 Boeegs

In [4]:
print(f"Total elements: {len(elements)}")

Total elements: 232


In [5]:
from collections import defaultdict

# --- group all elements by their type ---
grouped = defaultdict(list)
for el in elements:
    grouped[type(el).__name__].append(el)

# --- print distribution ---
print("\nElement type distribution:")
for type_name, els in sorted(grouped.items()):
    print(f"  {type_name:25s} : {len(els)}")

# --- display 5 samples per type ---
for type_name, els in sorted(grouped.items()):
    print(f"\n{'='*60}")
    print(f"TYPE: {type_name}  (total: {len(els)})")
    print(f"{'='*60}")

    for i, el in enumerate(els[:5]):   # only first 5
        print(f"\n  -- sample #{i+1} --")
        print(f"  page        : {el.metadata.page_number}")
        print(f"  text        : {el.text[:150] if el.text else None}")

        # Table-specific
        if type_name == "Table":
            html = el.metadata.text_as_html
            print(f"  text_as_html: {html[:200] if html else None}")

        # Image-specific
        if type_name == "Image":
            print(f"  image_path  : {el.metadata.image_path}")
            print(f"  mime_type   : {el.metadata.image_mime_type}")

        # common metadata
        print(f"  parent_id   : {el.metadata.parent_id}")
        print(f"  category_d  : {el.metadata.category_depth}")



Element type distribution:
  FigureCaption             : 5
  Footer                    : 7
  Formula                   : 4
  Header                    : 3
  Image                     : 7
  ListItem                  : 43
  NarrativeText             : 81
  Table                     : 4
  Text                      : 52
  Title                     : 26

TYPE: FigureCaption  (total: 5)

  -- sample #1 --
  page        : 3
  text        : Figure 1: The Transformer - model architecture.
  parent_id   : None
  category_d  : None

  -- sample #2 --
  page        : 8
  text        : Table 2: The Transformer achieves better BLEU scores than previous state-of-the-art models on the English-to-German and English-to-French newstest2014
  parent_id   : None
  category_d  : None

  -- sample #3 --
  page        : 10
  text        : Table 4: The Transformer generalizes well to English constituency parsing (Results are on Section 23 of WSJ)
  parent_id   : f1e4ca1b21d5040a5dbaa7caa0f7f963
  category_d  

In [6]:
# Tesseract Open Source OCR Engine (main repository)
# tesseract-ocr.github.io/

In [7]:
from unstructured.chunking.title import chunk_by_title

chunks = chunk_by_title(
    elements,                          # now contains Image elements too
    max_characters=1000,
    include_orig_elements=True,
)

for i, chunk in enumerate(chunks):
    orig = chunk.metadata.orig_elements or []
    type_dist = {}
    for el in orig:
        t = type(el).__name__
        type_dist[t] = type_dist.get(t, 0) + 1
    print(f"Chunk #{i+1:03d} | chars={len(chunk.text):5d} | page={chunk.metadata.page_number} | types={type_dist}")

Chunk #001 | chars=  708 | page=1 | types={'Text': 20, 'Header': 1, 'NarrativeText': 8, 'Title': 3}
Chunk #002 | chars=  993 | page=1 | types={'NarrativeText': 1}
Chunk #003 | chars=  146 | page=1 | types={'NarrativeText': 1}
Chunk #004 | chars=  988 | page=1 | types={'NarrativeText': 1, 'Text': 2}
Chunk #005 | chars=  524 | page=1 | types={'NarrativeText': 2, 'Title': 1}
Chunk #006 | chars=  755 | page=2 | types={'NarrativeText': 1}
Chunk #007 | chars=  733 | page=2 | types={'NarrativeText': 2}
Chunk #008 | chars=  839 | page=2 | types={'Title': 1, 'NarrativeText': 1}
Chunk #009 | chars=  991 | page=2 | types={'NarrativeText': 3}
Chunk #010 | chars=  871 | page=2 | types={'Title': 1, 'NarrativeText': 1, 'Footer': 1, 'Image': 1, 'FigureCaption': 1}
Chunk #011 | chars=  877 | page=3 | types={'NarrativeText': 2, 'Title': 1}
Chunk #012 | chars=  685 | page=3 | types={'NarrativeText': 1}
Chunk #013 | chars=  580 | page=3 | types={'Title': 1, 'NarrativeText': 3, 'Footer': 1, 'Text': 2, 'Ima

In [8]:
from pathlib import Path
from abc import ABC, abstractmethod

class BasePartitioner(ABC):
    @abstractmethod
    def partition(self, filepath: str)->list:
        pass

class PartitionerRegistry:
    registry: dict[str, type[BasePartitioner]] = {}
    pass

    @classmethod
    def register(cls, *extensions: str):
        def decorator(partition_cls: type[BasePartitioner]):
            for ext in extensions:
                cls.registry[ext.lower()] = partition_cls
            return partition_cls
        return decorator
    

    @classmethod
    def get(cls, filepath: str) -> BasePartitioner:
        ext = Path(filepath).suffix.lower()
        partitioner_cls = cls.registry.get(ext)
        if not partitioner_cls:
            raise ValueError(f"No partitioner registered for extension: {ext}")
        return partitioner_cls()

In [9]:
from pathlib import Path
from abc import ABC, abstractmethod

from unstructured.partition.pdf import partition_pdf
from unstructured.partition.docx import partition_docx
from unstructured.partition.doc import partition_doc
from unstructured.partition.pptx import partition_pptx
from unstructured.partition.xlsx import partition_xlsx
from unstructured.partition.csv import partition_csv
from unstructured.partition.tsv import partition_tsv
from unstructured.partition.html import partition_html
from unstructured.partition.text import partition_text
from unstructured.partition.md import partition_md
from unstructured.partition.xml import partition_xml
from unstructured.partition.image import partition_image
from unstructured.partition.email import partition_email
from unstructured.partition.msg import partition_msg
from unstructured.partition.epub import partition_epub


class BasePartitioner(ABC):
    @abstractmethod
    def partition(self, filepath: str) -> list:
        pass


class PartitionerRegistry:
    _registry: dict[str, type[BasePartitioner]] = {}

    @classmethod
    def register(cls, *extensions: str):
        def decorator(partitioner_cls: type[BasePartitioner]):
            for ext in extensions:
                cls._registry[ext.lower()] = partitioner_cls
            return partitioner_cls
        return decorator

    @classmethod
    def get(cls, filepath: str) -> BasePartitioner:
        ext = Path(filepath).suffix.lower()
        partitioner_cls = cls._registry.get(ext)
        if not partitioner_cls:
            raise ValueError(f"No partitioner registered for extension: {ext}")
        return partitioner_cls()


@PartitionerRegistry.register(".pdf")
class PDFPartitioner(BasePartitioner):
    def partition(self, filepath: str) -> list:
        return partition_pdf(filename=filepath, strategy="fast")


@PartitionerRegistry.register(".docx")
class DocxPartitioner(BasePartitioner):
    def partition(self, filepath: str) -> list:
        return partition_docx(filename=filepath)


@PartitionerRegistry.register(".doc")
class DocPartitioner(BasePartitioner):
    def partition(self, filepath: str) -> list:
        return partition_doc(filename=filepath)


@PartitionerRegistry.register(".pptx")
class PptxPartitioner(BasePartitioner):
    def partition(self, filepath: str) -> list:
        return partition_pptx(filename=filepath)


@PartitionerRegistry.register(".xlsx")
class XlsxPartitioner(BasePartitioner):
    def partition(self, filepath: str) -> list:
        return partition_xlsx(filename=filepath, include_header=True)


@PartitionerRegistry.register(".csv")
class CSVPartitioner(BasePartitioner):
    def partition(self, filepath: str) -> list:
        return partition_csv(filename=filepath)


@PartitionerRegistry.register(".tsv")
class TSVPartitioner(BasePartitioner):
    def partition(self, filepath: str) -> list:
        return partition_tsv(filename=filepath)


@PartitionerRegistry.register(".html", ".htm")
class HTMLPartitioner(BasePartitioner):
    def partition(self, filepath: str) -> list:
        return partition_html(filename=filepath)


@PartitionerRegistry.register(".txt")
class TextPartitioner(BasePartitioner):
    def partition(self, filepath: str) -> list:
        return partition_text(filename=filepath)


@PartitionerRegistry.register(".md")
class MarkdownPartitioner(BasePartitioner):
    def partition(self, filepath: str) -> list:
        return partition_md(filename=filepath)


@PartitionerRegistry.register(".xml")
class XMLPartitioner(BasePartitioner):
    def partition(self, filepath: str) -> list:
        return partition_xml(filename=filepath)


@PartitionerRegistry.register(".png", ".jpg", ".jpeg", ".tiff", ".bmp")
class ImagePartitioner(BasePartitioner):
    def partition(self, filepath: str) -> list:
        return partition_image(filename=filepath, strategy="hi_res", languages=["eng"])


@PartitionerRegistry.register(".eml")
class EmailPartitioner(BasePartitioner):
    def partition(self, filepath: str) -> list:
        return partition_email(filename=filepath)


@PartitionerRegistry.register(".msg")
class MsgPartitioner(BasePartitioner):
    def partition(self, filepath: str) -> list:
        return partition_msg(filename=filepath)


@PartitionerRegistry.register(".epub")
class EpubPartitioner(BasePartitioner):
    def partition(self, filepath: str) -> list:
        return partition_epub(filename=filepath)


def partition_file(filepath: str) -> list:
    partitioner = PartitionerRegistry.get(filepath)
    return partitioner.partition(filepath)

In [10]:
def partition_file(filepath: str) -> list:
    partitioner = PartitionerRegistry.get(filepath)
    return partitioner.partition(filepath)

In [11]:
# elements = partition_file("attention.pdf")
# elements = partition_file("report.docx")
# elements = partition_file("data.csv")
# elements = partition_file("slides.pptx")
# elements = partition_file("scan.png")


# elements = partition_file(r"data\\raw\\math.png") # not working 
elements = partition_file(r"02-chunking-unstructured.md")

print(f"Total elements: {len(elements)}")
for el in elements[:5]:
    print(type(el).__name__, "->", el.text[:80])

Total elements: 84
Title -> Unstructured — Chunking Deep Dive
Text -> Sources: Official Docs · Context7 · Unstructured readthedocs 0.12.6 · Unstructur
Title -> How Unstructured Chunking Differs from Normal Chunking
NarrativeText -> Most chunking libraries (LangChain, LlamaIndex) start from raw text and split on
NarrativeText -> Normal approach: raw text → split by \n\n / tokens → chunks Unstructured: raw do


In [12]:
from pathlib import Path

# img_path = (Path.cwd() / ".." / "data" / "raw" / "claude_prompt.txt").resolve()
# file_path = (Path.cwd() / ".." / "data" / "raw" / "agent plan.docx").resolve()
file_path = (Path.cwd() / ".." / "data" / "raw" / "project-presentation.pptx").resolve()
print(file_path, file_path.exists())

elements = partition_file(str(file_path))


D:\PROJECTS\multimodal_rag\data\raw\project-presentation.pptx True


In [13]:
print(f"Total elements: {len(elements)}")
for el in elements[:5]:
    print(type(el).__name__, "->", el.text[:80])

Total elements: 140
Title -> BHARATI VIDYAPEETH’S COLLEGE OF ENGINEERING
Text -> PUNE–412115.
Title ->  DEPARTMENT OF COMPUTER ENGINEERING
Title -> ACADEMIC YEAR: 2024 -2025 
Title -> Title: Multi-Modal Deepfake Detection: Techniques and Integration for Enhanced M


In [14]:
meta = el.metadata

# ── Always present (when available) ──────────────────
print(meta.filename)            # "report.pdf"
print(meta.file_directory)      # "/home/user/docs"
print(meta.filetype)            # "application/pdf"
print(meta.page_number)         # 1, 2, 3 ...
print(meta.languages)           # ['eng']
print(meta.last_modified)       # "2024-01-15T10:30:00"

# ── Hierarchy / Structure ─────────────────────────────
print(meta.parent_id)           # ID of the parent element (e.g. Title above this)
print(meta.category_depth)      # depth in document hierarchy (0 = top level)


# ── Table specific ────────────────────────────────────
print(meta.text_as_html)        # "<table><tr><td>...</td></tr></table>"
                                # only present for Table elements

# ── Email specific ────────────────────────────────────
print(meta.sent_from)           # ["sender@example.com"]
print(meta.sent_to)             # ["recipient@example.com"]
print(meta.subject)             # "Meeting Notes"
print(meta.email_message_id)    # "<unique-id@mail.com>"

# ── Image specific ────────────────────────────────────
print(meta.image_path)          # path to extracted image file
print(meta.image_mime_type)     # "image/png"

# ── Full dict ─────────────────────────────────────────
# print(meta.to_dict())           # everything as a flat dictionary

project-presentation.pptx
D:\PROJECTS\multimodal_rag\data\raw
application/vnd.openxmlformats-officedocument.presentationml.presentation
1
['eng']
2026-03-12T10:49:15
None
1
None
None
None
None
None
None
None


In [15]:
from collections import defaultdict
from pathlib import Path

file_path = (Path.cwd() / ".." / "data" / "raw" / "project-presentation.pptx").resolve()
print(file_path, file_path.exists())

elements = partition_file(str(file_path))

# --- group all elements by type ---
grouped = defaultdict(list)
for el in elements:
    grouped[type(el).__name__].append(el)

# --- distribution overview ---
print(f"\nTotal elements: {len(elements)}")
print("\nType distribution:")
for type_name, els in sorted(grouped.items()):
    print(f"  {type_name:25s} : {len(els)}")

# --- 5 samples per type ---
for type_name, els in sorted(grouped.items()):
    print(f"\n{'='*60}")
    print(f"TYPE: {type_name}  (total: {len(els)})")
    print(f"{'='*60}")

    for i, el in enumerate(els[:5]):
        print(f"\n  -- sample #{i+1} --")
        print(f"  page / slide    : {el.metadata.page_number}")
        print(f"  text            : {el.text[:150] if el.text else None}")

        if type_name == "Table":
            html = el.metadata.text_as_html
            print(f"  text_as_html    : {html[:300] if html else None}")

        if type_name == "Image":
            print(f"  image_path      : {el.metadata.image_path}")
            print(f"  mime_type       : {el.metadata.image_mime_type}")

        print(f"  parent_id       : {el.metadata.parent_id}")
        print(f"  category_depth  : {el.metadata.category_depth}")

D:\PROJECTS\multimodal_rag\data\raw\project-presentation.pptx True

Total elements: 140

Type distribution:
  ListItem                  : 18
  NarrativeText             : 28
  PageBreak                 : 27
  Text                      : 3
  Title                     : 64

TYPE: ListItem  (total: 18)

  -- sample #1 --
  page / slide    : 4
  text            : Deepfakes are AI-generated synthetic media that appear authentic but are artificially created or manipulated.
  parent_id       : bd60622b5b465da06fa1f90ef7bde0e2
  category_depth  : 1

  -- sample #2 --
  page / slide    : 4
  text            : Created using advanced machine learning techniques like GANs to produce convincing digital forgeries.
  parent_id       : bd60622b5b465da06fa1f90ef7bde0e2
  category_depth  : 1

  -- sample #3 --
  page / slide    : 4
  text            : Offers benefits in entertainment and education, while posing risks for misinformation, fraud, and identity theft.
  parent_id       : bd60622b5b465da06fa1

In [16]:
import os
from pathlib import Path
from pptx import Presentation
from pptx.util import Inches
from PIL import Image
import io

from unstructured.partition.pptx import partition_pptx
from unstructured.partition.image import partition_image


def extract_pptx_with_images(filepath: str, output_dir: str = "./slide_images") -> list:
    os.makedirs(output_dir, exist_ok=True)

    # --- Step 1: get text elements from pptx normally ---
    text_elements = partition_pptx(filename=filepath)
    print(f"Text elements extracted: {len(text_elements)}")

    # --- Step 2: convert each slide to image using python-pptx + PIL ---
    # requires: pip install python-pptx pillow
    prs = Presentation(filepath)
    slide_image_paths = []

    for slide_num, slide in enumerate(prs.slides, start=1):
        # export slide as PNG via LibreOffice (most reliable on Windows)
        # alternative: use aspose.slides (paid) or comtypes (Windows COM)
        slide_path = os.path.join(output_dir, f"slide_{slide_num:03d}.png")
        slide_image_paths.append((slide_num, slide_path))

    # --- Step 3: partition each slide image to get Image elements ---
    image_elements = []
    for slide_num, slide_path in slide_image_paths:
        if not os.path.exists(slide_path):
            print(f"Slide image not found: {slide_path} — skipping")
            continue
        els = partition_image(
            filename=slide_path,
            strategy="hi_res",
            languages=["eng"],
        )
        for el in els:
            el.metadata.page_number = slide_num   # tag with slide number
        image_elements.extend(els)
        print(f"Slide {slide_num}: {len(els)} elements extracted from image")

    return text_elements + image_elements

In [18]:
# inventory.xlsx
from pathlib import Path

file_path = (Path.cwd() / ".." / "data" / "raw" / "inventory.xlsx").resolve()
print(file_path, file_path.exists())

D:\PROJECTS\multimodal_rag\data\raw\inventory.xlsx True


In [19]:
elements = partition_file(str(file_path))

print(f"Total elements: {len(elements)}")
for el in elements[:5]:
    print(type(el).__name__, "->", el.text[:80])

Total elements: 2
Table -> Product Category Price Stock Description Laptop Electronics 999.99 50 High-perfo
Table -> Category Total_Items Total_Value Electronics 3 1389.97 Accessories 2 109.98


In [20]:
from collections import defaultdict

grouped = defaultdict(list)
for el in elements:
    grouped[type(el).__name__].append(el)

print("\nType distribution:")
for type_name, els in sorted(grouped.items()):
    print(f"  {type_name:20s} : {len(els)}")


Type distribution:
  Table                : 2


In [21]:
for i, el in enumerate(elements):
    print(f"\n{'='*60}")
    print(f"Element #{i+1} | type={type(el).__name__}")
    print(f"{'='*60}")

    # xlsx-specific metadata
    print(f"  sheet name    : {el.metadata.page_name}")      # which sheet
    print(f"  page_number   : {el.metadata.page_number}")    # sheet index
    print(f"  filename      : {el.metadata.filename}")
    print(f"  filetype      : {el.metadata.filetype}")

    # content
    print(f"\n  text (plain)  :\n    {el.text[:300] if el.text else None}")

    # HTML table — the most useful for xlsx
    html = el.metadata.text_as_html
    print(f"\n  text_as_html  :\n    {html[:500] if html else None}")


Element #1 | type=Table
  sheet name    : Products
  page_number   : 1
  filename      : inventory.xlsx
  filetype      : application/vnd.openxmlformats-officedocument.spreadsheetml.sheet

  text (plain)  :
    Product Category Price Stock Description Laptop Electronics 999.99 50 High-performance laptop with 16GB RAM and 512GB SSD Mouse Accessories 29.99 200 Wireless optical mouse with ergonomic design Keyboard Accessories 79.99 150 Mechanical keyboard with RGB backlighting Monitor Electronics 299.99 75 27

  text_as_html  :
    <table><tr><td>Product</td><td>Category</td><td>Price</td><td>Stock</td><td>Description</td></tr><tr><td>Laptop</td><td>Electronics</td><td>999.99</td><td>50</td><td>High-performance laptop with 16GB RAM and 512GB SSD</td></tr><tr><td>Mouse</td><td>Accessories</td><td>29.99</td><td>200</td><td>Wireless optical mouse with ergonomic design</td></tr><tr><td>Keyboard</td><td>Accessories</td><td>79.99</td><td>150</td><td>Mechanical keyboard with RGB backlighting</t

In [22]:
import pandas as pd

for i, el in enumerate(elements):
    html = el.metadata.text_as_html
    if not html:
        print(f"Element #{i+1} — no HTML table")
        continue

    print(f"\n--- Sheet: {el.metadata.page_name} | Element #{i+1} ---")

    try:
        # pd.read_html parses the HTML table directly
        dfs = pd.read_html(html)
        df = dfs[0]
        print(f"  shape  : {df.shape}  (rows x cols)")
        print(f"  columns: {list(df.columns)}")
        print(df.head(5).to_string(index=False))
    except Exception as e:
        print(f"  could not parse as DataFrame: {e}")
        print(f"  raw text: {el.text[:200]}")


--- Sheet: Products | Element #1 ---
  shape  : (6, 5)  (rows x cols)
  columns: [0, 1, 2, 3, 4]
       0           1      2     3                                                   4
 Product    Category  Price Stock                                         Description
  Laptop Electronics 999.99    50 High-performance laptop with 16GB RAM and 512GB SSD
   Mouse Accessories  29.99   200        Wireless optical mouse with ergonomic design
Keyboard Accessories  79.99   150           Mechanical keyboard with RGB backlighting
 Monitor Electronics 299.99    75                 27-inch 4K monitor with HDR support

--- Sheet: Summary | Element #2 ---
  shape  : (3, 3)  (rows x cols)
  columns: [0, 1, 2]
          0           1           2
   Category Total_Items Total_Value
Electronics           3     1389.97
Accessories           2      109.98


C:\Users\Admin\AppData\Local\Temp\ipykernel_7664\1857703558.py:13: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(html)
C:\Users\Admin\AppData\Local\Temp\ipykernel_7664\1857703558.py:13: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(html)


In [24]:
from collections import defaultdict
import pandas as pd
from io import StringIO

# Group elements by sheet name
by_sheet = defaultdict(list)
for el in elements:
    sheet = el.metadata.page_name or "unknown"
    by_sheet[sheet].append(el)

# Print summary header
print(f"Total sheets: {len(by_sheet)}")

# Iterate through each sheet and its sub-tables
for sheet_name, elements_in_sheet in by_sheet.items():
    print(f"  Sheet: '{sheet_name}' | sub-tables found: {len(elements_in_sheet)}")
    
    for j, el in enumerate(elements_in_sheet):
        if el.metadata.text_as_html:
            try:
                # Correct way to parse HTML string in modern pandas
                df = pd.read_html(StringIO(el.metadata.text_as_html))[0]
                shape_str = str(df.shape)
            except Exception as e:
                shape_str = f"parse error: {str(e)}"
        else:
            df = None
            shape_str = "no html"

        # Preview first 60 characters of the extracted text
        preview = el.text[:60].replace("\n", " ").strip() if el.text else ""
        
        print(f"    sub-table #{j+1} | shape={shape_str} | text_preview: {preview}")

Total sheets: 2
  Sheet: 'Products' | sub-tables found: 1
    sub-table #1 | shape=(6, 5) | text_preview: Product Category Price Stock Description Laptop Electronics
  Sheet: 'Summary' | sub-tables found: 1
    sub-table #1 | shape=(3, 3) | text_preview: Category Total_Items Total_Value Electronics 3 1389.97 Acces
